In [2]:
import pandas as pd
import os
from typing import List, Dict, Tuple

# Install required packages if not already installed
try:
    import fitz  # PyMuPDF
except ImportError:
    print("Installing PyMuPDF...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "PyMuPDF"])
    import fitz

def extract_pdf_data(pdf_path: str) -> Tuple[str, Dict[int, str], fitz.Document]:
    """Extract text from PDF and return full text, page texts, and document."""
    try:
        doc = fitz.open(pdf_path)
        full_text, page_texts = "", {}
        for i in range(doc.page_count):
            page_text = doc[i].get_text()
            page_texts[i + 1] = page_text.lower()
            full_text += page_text
        return full_text.lower(), page_texts, doc
    except Exception as e:
        print(f"Error reading PDF: {e}")
        return "", {}, None

def find_and_highlight_matches(requirement: str, page_texts: Dict[int, str], doc: fitz.Document, context: int = 200) -> Dict:
    """Find requirement matches, extract snippets, and highlight in PDF."""
    req_lower = requirement.lower()
    pages_found, snippets, highlighted_pages = [], {}, []
    
    for page_num, page_text in page_texts.items():
        if req_lower not in page_text:
            continue
            
        pages_found.append(page_num)
        page_snippets = []
        page = doc[page_num - 1]
        original_text = page.get_text()
        
        # Find all occurrences and create snippets
        start = 0
        while True:
            pos = page_text.find(req_lower, start)
            if pos == -1:
                break
                
            # Extract snippet with context
            snippet_start = max(0, pos - context)
            snippet_end = min(len(page_text), pos + len(req_lower) + context)
            snippet = ' '.join(page_text[snippet_start:snippet_end].split())
            
            if snippet_start > 0:
                snippet = "..." + snippet
            if snippet_end < len(page_text):
                snippet += "..."
            page_snippets.append(snippet)
            
            # Highlight in PDF
            snippet_text = original_text[snippet_start:snippet_end]
            text_instances = page.search_for(snippet_text.strip(), flags=fitz.TEXT_DEHYPHENATE)
            
            if not text_instances:
                # Fallback: find requirement and expand highlight area
                req_instances = page.search_for(requirement, flags=fitz.TEXT_DEHYPHENATE)
                if not req_instances:
                    orig_pos = original_text.lower().find(req_lower)
                    if orig_pos != -1:
                        orig_req = original_text[orig_pos:orig_pos + len(requirement)]
                        req_instances = page.search_for(orig_req, flags=fitz.TEXT_DEHYPHENATE)
                
                for rect in req_instances:
                    expanded_rect = fitz.Rect(
                        max(0, rect.x0 - 100), max(0, rect.y0 - 30),
                        min(page.rect.width, rect.x1 + 100), min(page.rect.height, rect.y1 + 30)
                    )
                    highlight = page.add_highlight_annot(expanded_rect)
                    highlight.set_colors(stroke=[1, 1, 0])
                    highlight.update()
            else:
                for inst in text_instances:
                    highlight = page.add_highlight_annot(inst)
                    highlight.set_colors(stroke=[1, 1, 0])
                    highlight.update()
            
            start = pos + 1
        
        if page_snippets:
            snippets[page_num] = page_snippets
            if page_num not in highlighted_pages:
                highlighted_pages.append(page_num)
    
    return {
        'exact_match': bool(pages_found),
        'exact_pages': pages_found,
        'exact_snippets': snippets,
        'highlighted_pages': highlighted_pages
    }

def check_document_requirements(document_id: int, csv_path: str, documents_folder: str, output_folder: str):
    """Main function to check requirements and generate outputs."""
    # Load data
    try:
        df = pd.read_csv(csv_path)
        doc_requirements = df[df['document_id'] == document_id].copy()
        if doc_requirements.empty:
            print(f"No requirements found for document_id {document_id}")
            return
    except Exception as e:
        print(f"Error reading CSV: {e}")
        return
    
    # Load PDF
    pdf_path = os.path.join(documents_folder, f"{document_id}.pdf")
    if not os.path.exists(pdf_path):
        print(f"PDF not found: {pdf_path}")
        return
    
    pdf_text, page_texts, doc = extract_pdf_data(pdf_path)
    if not doc:
        return
    
    print(f"Processing {len(doc_requirements)} requirements from {len(page_texts)} pages...")
    
    # Process requirements
    results = []
    any_highlights = False
    
    for idx, row in doc_requirements.iterrows():
        requirement = str(row['requirement'])
        if pd.isna(requirement) or not requirement.strip():
            continue
        
        result = find_and_highlight_matches(requirement, page_texts, doc)
        if result['highlighted_pages']:
            any_highlights = True
        
        # Format snippets
        snippets_text = ""
        for page_num in sorted(result['exact_snippets'].keys()):
            snippets_text += f"Page {page_num}: "
            snippets_text += " | ".join(f'"{s}"' for s in result['exact_snippets'][page_num])
            snippets_text += "\n"
        
        results.append({
            'row_index': idx, 'document_id': document_id,
            'model_name': row.get('model_name', ''),
            'constraint_type': row.get('constraint_type', ''),
            'scope': row.get('scope', ''),
            'numerical_value': row.get('numerical_value', ''),
            'unit': row.get('unit', ''),
            'requirement': requirement,
            'exact_match': result['exact_match'],
            'pages_found': ', '.join(map(str, result['exact_pages'])),
            'highlighted_pages': ', '.join(map(str, result['highlighted_pages'])),
            'matching_text_snippets': snippets_text.strip()
        })
    
    results_df = pd.DataFrame(results)
    
    # Save outputs
    os.makedirs(output_folder, exist_ok=True)
    
    if any_highlights:
        highlighted_pdf = os.path.join(output_folder, f"document_{document_id}_highlighted.pdf")
        doc.save(highlighted_pdf)
        print(f"Highlighted PDF: {highlighted_pdf}")
    
    doc.close()
    
    # Statistics
    total = len(results_df)
    matches = results_df['exact_match'].sum()
    highlighted = results_df[results_df['highlighted_pages'] != ''].shape[0]
    
    print(f"\n=== SUMMARY ===")
    print(f"Total: {total}, Found: {matches}, Highlighted: {highlighted}, Success: {matches/total*100:.1f}%")
    
    # Save results
    results_df.to_csv(os.path.join(output_folder, f"document_{document_id}_results.csv"), index=False)
    
    # Save summary
    with open(os.path.join(output_folder, f"document_{document_id}_summary.txt"), 'w') as f:
        f.write(f"SUMMARY FOR DOCUMENT {document_id}\n{'='*50}\n")
        f.write(f"Total: {total}, Found: {matches}, Success: {matches/total*100:.1f}%\n\n")
        
        for _, row in results_df.iterrows():
            status = "✓" if row['exact_match'] else "✗"
            f.write(f"{status} {row['requirement'][:80]}...\n")
            if row['matching_text_snippets']:
                f.write(f"   {row['matching_text_snippets'][:100]}...\n")
            f.write("\n")
    
    print(f"Results saved to: {output_folder}")

In [6]:
# Configuration and execution
DOCUMENT_ID = 11
CSV_PATH = "/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/Extract_regulations/Filtered_regulations/combined_all_filtered_constraints.csv"
DOCUMENTS_FOLDER = "/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/Documents"
OUTPUT_FOLDER = "/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/Extract_regulations/Check_doc_result"

if __name__ == "__main__":
    check_document_requirements(DOCUMENT_ID, CSV_PATH, DOCUMENTS_FOLDER, OUTPUT_FOLDER)

Processing 269 requirements from 454 pages...
Highlighted PDF: /Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/Extract_regulations/Check_doc_result/document_11_highlighted.pdf

=== SUMMARY ===
Total: 269, Found: 32, Highlighted: 32, Success: 11.9%
Results saved to: /Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/Extract_regulations/Check_doc_result
